# EDA: Calidad de Red

## Caso Práctico - Empresa de Telecomunicaciones
## MBI — Prácticas Aplicadas 2026

---

En este notebook analizamos la tabla de calidad de señal por zona y mes.

A diferencia de las otras fuentes, esta tabla **no está al nivel del cliente** sino al nivel de zona-mes. Esto significa que no podemos saber exactamente qué calidad de red experimenta cada cliente individualmente, pero sí podemos saber la calidad de la zona donde vive.

Según el profesor, la calidad de red es uno de los drivers principales del churn:
- Mala calidad → menos consumo
- Mala calidad → más impagos
- Mala calidad → más llamadas a soporte
- Todo eso → más churn

**Fuente:** `calidad_senal_zona_mensual.csv`

**Variables:**
- `cobertura_4g_pct` / `cobertura_5g_pct` → % de cobertura por tecnología
- `latencia_ms` → tiempo de respuesta de la red (ms)
- `velocidad_media_mbps` → velocidad media de descarga
- `tasa_cortes_pct` → % de llamadas/conexiones con corte
- `indice_calidad_global` → índice sintético de calidad (más alto = mejor)
- `incidencia_masiva` → si hubo una incidencia masiva ese mes en esa zona


## Objetivos

1. Explorar la estructura y calidad del dataset
2. Detectar problemas de calidad de datos (errores tipográficos, outliers, nulos)
3. Contrastar hipótesis sobre la calidad de red:
   - **H1**: ¿Las zonas rurales tienen peor calidad de red?
   - **H2**: ¿Hay regiones con peor calidad estructural?
   - **H3**: ¿La cobertura 5G ha mejorado con el tiempo?
   - **H4**: ¿Las incidencias masivas se asocian a picos de latencia y cortes?
   - **H5**: ¿La calidad de red de la zona está relacionada con el churn del cliente?
4. Clasificación simple para validar H5 cuantitativamente


## Nota metodológica

Para H5 necesitamos cruzar esta tabla con los clientes y el churn. El cruce se hace por `zona_id`: cada cliente tiene asignada una zona en `clientes.csv`, y usamos la calidad media de esa zona como proxy de la experiencia del cliente.

Es una aproximación, porque dos clientes en la misma zona pueden tener experiencias distintas (distintas calles, distintos dispositivos...), pero es la mejor que podemos hacer con los datos disponibles.


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

# Añadimos la raíz del proyecto al path para poder importar src/
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.utils.helpers import (
    resumen_df, test_mannwhitney, cramers_v,
    boxplot_churn, kde_churn, barras_churn_cat, perfil_churner
)

PAL = {'No Churn': '#4C9BE8', 'Churn': '#E85C4C'}
print('Librerías cargadas')

## 2. Carga de datos


In [ ]:
# Rutas relativas desde notebooks/
PATH_CALIDAD   = ROOT / 'data' / 'raw' / 'calidad_senal_zona_mensual.csv'
PATH_CLIENTES  = ROOT / 'data' / 'raw' / 'clientes.csv'
PATH_CHURN     = ROOT / 'data' / 'raw' / 'churn_target.csv'

calidad  = pd.read_csv(PATH_CALIDAD)
clientes = pd.read_csv(PATH_CLIENTES)
churn    = pd.read_csv(PATH_CHURN)

calidad['fecha'] = pd.to_datetime(calidad['fecha'], format='mixed', dayfirst=True)
churn['fecha']   = pd.to_datetime(churn['fecha'],   format='mixed', dayfirst=True)

print(f'calidad:  {calidad.shape[0]:,} filas x {calidad.shape[1]} columnas')
print(f'clientes: {clientes.shape[0]:,} filas x {clientes.shape[1]} columnas')
print(f'churn:    {churn.shape[0]:,} filas x {churn.shape[1]} columnas')

---
## 3. Exploración inicial y calidad de datos

Antes de cualquier hipótesis, miramos qué tenemos y si hay problemas.


In [ ]:
resumen_df(calidad, 'calidad_senal_zona_mensual.csv')
display(calidad.head())

In [ ]:
display(calidad.describe())

### 3.1 Problemas de calidad detectados

Mirando el describe ya se ven cosas raras que hay que investigar antes de seguir.


In [ ]:
# 1. Tipos de zona: ¿hay errores tipográficos?
print('Valores únicos en tipo_zona:')
print(calidad['tipo_zona'].value_counts())
print()

# 2. Velocidad negativa — imposible físicamente
print(f'Registros con velocidad_media_mbps < 0: {(calidad["velocidad_media_mbps"] < 0).sum()}')
print(calidad[calidad['velocidad_media_mbps'] < 0][['fecha','zona_id','tipo_zona','velocidad_media_mbps']])
print()

# 3. Latencia extrema (> 200ms es ya muy alta en una red móvil)
print(f'Registros con latencia > 200ms: {(calidad["latencia_ms"] > 200).sum()}')
print(calidad[calidad['latencia_ms'] > 200][['fecha','zona_id','tipo_zona','latencia_ms']])
print()

# 4. Nulos
print('Nulos por columna:')
print(calidad.isnull().sum())

### Hallazgos de calidad

**Errores tipográficos en `tipo_zona`**: aparecen valores como `suburbanx`, `urbana??` y `rural-1` que claramente son errores de introducción de datos. Los tratamos como categorías incorrectas y los normalizamos.

**Velocidad negativa**: hay registros con `velocidad_media_mbps < 0`, lo cual es físicamente imposible. Probablemente son errores del sistema de medición. Los trataremos como nulos.

**Latencia extrema**: hay un registro con más de 500ms de latencia, que es un outlier claro. Puede ser una incidencia puntual real o un error de medición.

Estos problemas son importantes de documentar porque si los usamos sin limpiar, las correlaciones que calculemos estarán distorsionadas.


In [ ]:
calidad_clean = calidad.copy()

# Normalizar tipo_zona
mapa_zona = {
    'suburbanx':    'suburbana',
    'urbana??':     'urbana_premium',
    'rural-1':      'rural',
}
calidad_clean['tipo_zona'] = calidad_clean['tipo_zona'].replace(mapa_zona)
print('Tipos de zona tras limpieza:', calidad_clean['tipo_zona'].unique())

# Velocidad negativa → NaN
mask_vel = calidad_clean['velocidad_media_mbps'] < 0
calidad_clean.loc[mask_vel, 'velocidad_media_mbps'] = np.nan
print(f'Velocidades negativas corregidas: {mask_vel.sum()}')

# Latencia > 300ms → NaN (umbral conservador)
mask_lat = calidad_clean['latencia_ms'] > 300
calidad_clean.loc[mask_lat, 'latencia_ms'] = np.nan
print(f'Latencias extremas corregidas: {mask_lat.sum()}')

### 3.2 Distribución de las variables de calidad


In [ ]:
vars_num = ['cobertura_4g_pct', 'cobertura_5g_pct', 'latencia_ms',
            'velocidad_media_mbps', 'tasa_cortes_pct', 'indice_calidad_global']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(vars_num):
    datos = calidad_clean[col].dropna()
    axes[i].hist(datos, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(datos.mean(), color='red', linestyle='--',
                    label=f'Media: {datos.mean():.1f}')
    axes[i].axvline(datos.median(), color='orange', linestyle='--',
                    label=f'Mediana: {datos.median():.1f}')
    axes[i].set_title(f'Distribución de {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de variables de calidad de red', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Análisis de Hipótesis


### H1 — ¿Las zonas rurales tienen peor calidad de red?

Es lógico esperar que las zonas rurales tengan menos infraestructura de red. Si esto se confirma, el tipo de zona será una variable importante para explicar el churn en esas áreas.


In [ ]:
orden_zonas = ['rural', 'suburbana', 'urbana_premium']
vars_calidad = ['indice_calidad_global', 'cobertura_4g_pct',
                'cobertura_5g_pct', 'latencia_ms',
                'velocidad_media_mbps', 'tasa_cortes_pct']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, var in enumerate(vars_calidad):
    data_plot = calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
    sns.boxplot(data=data_plot, x='tipo_zona', y=var,
                order=orden_zonas,
                palette=sns.color_palette('Blues', 3),
                ax=axes[i])
    axes[i].set_title(f'{var} por tipo de zona', fontweight='bold')
    axes[i].set_xlabel('Tipo de zona')
    axes[i].set_ylabel(var)

plt.suptitle('H1 — Calidad de red por tipo de zona', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen
resumen_zonas = (calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
                 .groupby('tipo_zona')[vars_calidad]
                 .mean()
                 .reindex(orden_zonas)
                 .round(2))
display(resumen_zonas)

### Resultado H1

Completar tras ejecutar.

**Pistas para interpretar**: Si los boxplots muestran que rural tiene peor índice de calidad global, más latencia y más tasa de cortes que urbana_premium, la hipótesis se confirma. Fíjate también en si la cobertura 5G es prácticamente nula en rural — eso sería un dato muy visual para la presentación.


### H2 — ¿Hay regiones con peor calidad estructural?

Más allá del tipo de zona, puede haber diferencias geográficas entre regiones: Norte, Sur, Este, Oeste, Centro. Si alguna región tiene peor infraestructura de forma sistemática, eso explicaría por qué tiene más churn.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Índice de calidad global por región
calidad_region = (calidad_clean.groupby('region')['indice_calidad_global']
                  .mean()
                  .sort_values())

bars = axes[0].barh(calidad_region.index, calidad_region.values,
                    color=sns.color_palette('RdYlGn', len(calidad_region)))
axes[0].set_title('Índice de calidad global medio por región', fontweight='bold')
axes[0].set_xlabel('Índice de calidad global (media)')
for bar, val in zip(bars, calidad_region.values):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# Heatmap: calidad media por región x tipo_zona
pivot = (calidad_clean[calidad_clean['tipo_zona'].isin(orden_zonas)]
         .groupby(['region', 'tipo_zona'])['indice_calidad_global']
         .mean()
         .unstack())
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Calidad global media — Región x Tipo de zona', fontweight='bold')
axes[1].set_xlabel('Tipo de zona')
axes[1].set_ylabel('Región')

plt.tight_layout()
plt.show()

### Resultado H2

Completar tras ejecutar.

**Pistas para interpretar**: El heatmap es especialmente útil aquí porque te permite ver si hay una región que sea mala en todos los tipos de zona, o si el problema es específico de zonas rurales en ciertas regiones. Una celda roja en rural+Centro sería una señal muy concreta para el negocio.


### H3 — ¿La cobertura 5G ha mejorado con el tiempo?

El dataset cubre varios años. El 5G es una tecnología que se ha ido desplegando progresivamente. Si vemos una tendencia creciente en `cobertura_5g_pct`, confirmaríamos que la empresa ha estado invirtiendo en mejorar su red. Esto también puede afectar al churn: los clientes en zonas donde llegó el 5G tarde pueden haber abandonado antes.


In [ ]:
evol = calidad_clean.groupby('fecha').agg(
    cobertura_5g      = ('cobertura_5g_pct',       'mean'),
    cobertura_4g      = ('cobertura_4g_pct',       'mean'),
    velocidad         = ('velocidad_media_mbps',   'mean'),
    latencia          = ('latencia_ms',            'mean'),
    calidad_global    = ('indice_calidad_global',  'mean'),
    tasa_cortes       = ('tasa_cortes_pct',        'mean'),
).reset_index()

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
axes = axes.flatten()

metricas = [
    ('cobertura_5g',   'Cobertura 5G (%)',          '#9b59b6'),
    ('cobertura_4g',   'Cobertura 4G (%)',          '#3498db'),
    ('velocidad',      'Velocidad media (Mbps)',     '#27ae60'),
    ('latencia',       'Latencia media (ms)',        '#e74c3c'),
    ('calidad_global', 'Índice calidad global',     '#f39c12'),
    ('tasa_cortes',    'Tasa de cortes (%)',         '#e67e22'),
]

for i, (col, titulo, color) in enumerate(metricas):
    axes[i].plot(evol['fecha'], evol[col], color=color, linewidth=2)
    axes[i].fill_between(evol['fecha'], evol[col], alpha=0.15, color=color)
    axes[i].set_title(titulo, fontweight='bold')
    axes[i].set_ylabel(titulo)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('H3 — Evolución temporal de las métricas de calidad de red',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Resultado H3

Completar tras ejecutar.

**Pistas para interpretar**: Si la línea del 5G sube de forma constante, la hipótesis se confirma. También es interesante ver si la latencia baja a medida que el 5G mejora (son variables inversamente relacionadas). Si ves picos bruscos en tasa_cortes o latencia, probablemente coincidan con las incidencias masivas que analizamos en H4.


### H4 — ¿Las incidencias masivas se asocian a picos de latencia y cortes?

Las incidencias masivas son eventos puntuales donde la red falla de forma generalizada en una zona. Es de esperar que en esos meses la latencia suba y la tasa de cortes también. Si esto se confirma, la variable `incidencia_masiva` es una señal importante para el modelo.


In [ ]:
calidad_clean['incidencia_label'] = calidad_clean['incidencia_masiva'].map(
    {0: 'Sin incidencia', 1: 'Con incidencia'}
)

vars_impacto = ['latencia_ms', 'tasa_cortes_pct',
                'velocidad_media_mbps', 'indice_calidad_global']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, var in enumerate(vars_impacto):
    data_plot = calidad_clean.dropna(subset=[var, 'incidencia_label'])
    sns.boxplot(data=data_plot, x='incidencia_label', y=var,
                order=['Sin incidencia', 'Con incidencia'],
                palette={'Sin incidencia': '#4C9BE8', 'Con incidencia': '#E85C4C'},
                ax=axes[i])
    axes[i].set_title(f'{var}\nvs incidencia masiva', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('H4 — Impacto de las incidencias masivas en la calidad de red',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla comparativa
print('\nMedia de métricas según incidencia masiva:')
display(calidad_clean.groupby('incidencia_masiva')[vars_impacto].mean().round(2))

### Resultado H4

Completar tras ejecutar.

**Pistas para interpretar**: Con solo 5 incidencias masivas en todo el dataset, los grupos son muy desiguales y los resultados hay que tomarlos con cautela. Aun así, si los boxplots muestran diferencias claras, es un resultado interesante. Si no las muestran, puede ser porque las incidencias están ya recogidas en el `stress_calidad_lag` de las otras tablas.


### H5 — ¿La calidad de red de la zona está relacionada con el churn del cliente?

Esta es la hipótesis más importante del notebook porque conecta la calidad de red directamente con el abandono. Para analizarla, cruzamos la calidad media por zona con el perfil del cliente y su etiqueta de churn.


In [ ]:
# Calidad media por zona (sobre todo el período)
calidad_zona = calidad_clean.groupby('zona_id').agg(
    calidad_global_media  = ('indice_calidad_global',  'mean'),
    calidad_global_min    = ('indice_calidad_global',  'min'),
    cobertura_4g_media    = ('cobertura_4g_pct',       'mean'),
    cobertura_5g_media    = ('cobertura_5g_pct',       'mean'),
    latencia_media        = ('latencia_ms',            'mean'),
    velocidad_media       = ('velocidad_media_mbps',   'mean'),
    tasa_cortes_media     = ('tasa_cortes_pct',        'mean'),
    n_incidencias_zona    = ('incidencia_masiva',      'sum'),
).reset_index()

# Target: ever_churn por cliente
churn_agg = churn.groupby('cliente_id')['churn'].max().reset_index()
churn_agg.columns = ['cliente_id', 'ever_churn']

# Tabla analítica: cliente + zona + calidad + churn
df = (clientes[['cliente_id', 'zona_id']]
      .merge(churn_agg,     on='cliente_id', how='inner')
      .merge(calidad_zona,  on='zona_id',    how='left'))

df['churn_label'] = df['ever_churn'].map({0: 'No Churn', 1: 'Churn'})

print(f'Tabla analítica: {df.shape[0]:,} clientes')
print(f'Ever churn: {df["ever_churn"].mean()*100:.1f}%')
display(df.head())

In [ ]:
vars_calidad_cliente = ['calidad_global_media', 'cobertura_5g_media',
                        'latencia_media', 'tasa_cortes_media']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, var in enumerate(vars_calidad_cliente):
    data_plot = df.dropna(subset=[var]).copy()
    sns.boxplot(data=data_plot, x='churn_label', y=var,
                order=['No Churn', 'Churn'],
                palette=PAL, ax=axes[i])
    axes[i].set_title(f'{var}\nvs Churn', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('H5 — Calidad de red de la zona vs Churn del cliente',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tests estadísticos
print('Tests Mann-Whitney:')
for var in vars_calidad_cliente:
    test_mannwhitney(df, var)

### Resultado H5

Completar tras ejecutar.

**Pistas para interpretar**: Si los clientes en zonas con peor calidad (menor índice global, mayor latencia, más cortes) tienen más churn, la hipótesis se confirma y la calidad de red es un predictor válido. Recuerda que aquí estamos usando la calidad media de la zona, no la individual del cliente — es una aproximación, y eso hay que mencionarlo al defender el trabajo.


---
## 5. Clasificación simple — Validación cuantitativa

Hasta aquí hemos analizado las hipótesis de forma visual y con tests estadísticos. Ahora vamos un paso más allá: entrenamos un modelo simple usando **solo las variables de calidad de red** para ver si son capaces de predecir el churn.

Usamos dos modelos:
- **Regresión logística**: modelo lineal, fácil de interpretar
- **Árbol de decisión**: captura relaciones no lineales

La métrica que usamos es el **AUC-ROC**: mide qué tan bien separa el modelo los churners de los no churners. Un AUC de 0.5 es igual que lanzar una moneda, 1.0 es perfecto.

⚠️ **Importante**: este modelo es solo para validar si las variables tienen poder predictivo. No es el modelo final del proyecto.


In [ ]:
vars_modelo = ['calidad_global_media', 'calidad_global_min', 'cobertura_4g_media',
               'cobertura_5g_media', 'latencia_media', 'velocidad_media',
               'tasa_cortes_media', 'n_incidencias_zona']

# Preparamos los datos
df_modelo = df.dropna(subset=vars_modelo + ['ever_churn'])
X = df_modelo[vars_modelo]
y = df_modelo['ever_churn']

print(f'Clientes para el modelo: {len(X):,}')
print(f'Tasa de churn: {y.mean()*100:.1f}%\n')

# Regresión logística
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=1000))
])
auc_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='roc_auc').mean()

# Árbol de decisión
pipe_dt = Pipeline([
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])
auc_dt = cross_val_score(pipe_dt, X, y, cv=5, scoring='roc_auc').mean()

print(f'AUC Regresión Logística: {auc_lr:.3f}')
print(f'AUC Árbol de Decisión:   {auc_dt:.3f}')

# Importancia de variables (árbol)
pipe_dt.fit(X, y)
importancias = pd.Series(
    pipe_dt.named_steps['model'].feature_importances_,
    index=vars_modelo
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importancia de variables — Árbol de decisión\n(solo variables de calidad de red)',
             fontweight='bold')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

### Resultado clasificación

Completar tras ejecutar.

**Cómo interpretar el AUC**:
- < 0.55 → las variables de calidad de red apenas ayudan a predecir el churn
- 0.55 - 0.65 → tienen algo de poder predictivo pero necesitan combinarse con otras fuentes
- > 0.65 → son variables relevantes por sí solas

Lo más probable es que el AUC sea moderado, porque la calidad de red es un factor importante pero no el único. La combinación con soporte, facturación y perfil del cliente debería mejorar bastante el resultado en el modelo final.


---
## 6. Conclusiones

**H1 — Calidad por tipo de zona:**
Completar tras ejecutar.

**H2 — Calidad por región:**
Completar tras ejecutar.

**H3 — Evolución temporal del 5G:**
Completar tras ejecutar.

**H4 — Incidencias masivas:**
Hay que tomar los resultados con cautela porque solo hay 5 incidencias masivas en todo el dataset. No es suficiente para sacar conclusiones robustas, pero sí es una señal que se refuerza con el `stress_calidad_lag` que ya vimos en las otras tablas.

**H5 — Calidad de red y churn:**
Completar tras ejecutar.

**Clasificación:**
Completar tras ejecutar.

### Variables más interesantes para el modelo final
- `indice_calidad_global` de la zona del cliente
- `cobertura_5g_pct` — diferencia entre zonas
- `tasa_cortes_pct` — experiencia directa del cliente
- `latencia_ms` — impacto en la percepción de calidad

---
*Notebook elaborado como parte de la asignatura de Prácticas Aplicadas — Máster en Ciencia de Datos 2026*
